<a href="https://colab.research.google.com/github/hhlubis/Analysis-SDGs/blob/main/%5B1%5D_Visualisasi_Capaian_SDGs_Wheel_Radial_Bar_Chart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Library

In [ ]:
import plotly.graph_objects as go
import numpy as np
import os
import base64
from google.colab import drive

# Konfigurasi ke Google Drive

In [ ]:
# 1. Hubungkan ke Google Drive
drive.mount('/content/drive')

# 2. Konfigurasi Data & Folder
icon_folder = "/content/drive/MyDrive/[3] Work/[2] SDGs-Bappeda DKI//Icon Goals SDGs/"

Mounted at /content/drive


# Input Data dan Warna Bar

In [ ]:
sdgs = [f"SDG {i}" for i in range(1, 18)]
values = [45.45, 50, 64.7, 30, 75, 100, 33.33, 80, 60, 100, 70, 100, 66.67, 75, 100, 84.21, 40]

colors = [
    '#E5243B', '#DDA63A', '#4C9F38', '#C5192D', '#FF3A21', '#26BDE2',
    '#FCC30B', '#A21942', '#FD6925', '#DD1367', '#FD9D24', '#BF8B2E',
    '#3F7E44', '#0A97D9', '#56C02B', '#00689D', '#19486A'
]

def get_base64_image(image_path):
    with open(image_path, "rb") as img_file:
        return "data:image/png;base64," + base64.b64encode(img_file.read()).decode('utf-8')

# Pembuatan Layer Visual (Radial Bar & Pita)

In [ ]:
# 3. Menghitung Sudut & Lebar Segmen (Agar tidak ada celah)

fig = go.Figure()
angles = np.linspace(0, 360, len(sdgs), endpoint=False)

width_per_segment = 360 / 19  # Membagi lingkaran sempurna menjadi 17 bagian



# --- LAYER 1: Bar Berwarna (Data Capaian) ---

fig.add_trace(go.Barpolar(

    r=values,

    theta=angles,

    width=width_per_segment,

    base=40,

    marker_color=colors,

    marker_line_color="white", # Garis pemisah tipis antar batang

    marker_line_width=0.5,

    name='Achievement'

)) # <-- Berhenti di sini (tanpa text/textposition)



# --- TAMBAHAN BARU: Layer Angka Capaian Persentase ---

# Menghitung posisi radius teks agar berada tepat di ujung luar batang (base 40 + nilai bar + offset 8)

text_r = [40 + v + 9.5 for v in values]

text_labels = [f"<b>{v}%</b>" for v in values] # Menggunakan tag <b> agar tulisan tebal



fig.add_trace(go.Scatterpolar(

    r=text_r,

    theta=angles,

    mode='text',

    text=text_labels,

    textposition='middle center',

    textfont=dict(

        size=11,
        color=colors,

        family='Arial'

    ),

    hoverinfo='skip',

    showlegend=False

))



# --- LAYER 2: PITA RADIAL (Background Label - Rapat Tanpa Celah) ---

fig.add_trace(go.Barpolar(

    r=[55]*17, # Ketebalan pita (r 45)

    theta=angles,

    width=width_per_segment, # Lebar penuh agar berhimpitan

    base=160, # Dimulai dari radius 155

    marker_color=colors,

    marker_line_color="white",

    marker_line_width=1, # Garis putih tipis antar kotak pita

    hoverinfo='skip'

))

# Pemrosesan Ikon SDGs

In [ ]:
# --- LAYER 4: Penempatan Ikon (Tepat di Tengah Pita Radial) ---
for i in range(len(sdgs)):
    theta_rad = np.deg2rad(90 - angles[i])

    # r_offset disesuaikan agar ikon berada di tengah pita (Radius ~177)
    r_offset = 0.435

    img_x = 0.5 + r_offset * np.cos(theta_rad)
    img_y = 0.5 + r_offset * np.sin(theta_rad)

    file_name = f"Goal-{str(i+1).zfill(2)}.png"
    file_path = os.path.join(icon_folder, file_name)

    if os.path.exists(file_path):
        try:
            encoded_image = get_base64_image(file_path)
            fig.add_layout_image(
                dict(
                    source=encoded_image,
                    xref="paper", yref="paper",
                    x=img_x, y=img_y,
                    sizex=0.070, sizey=0.070, # Ukuran ikon di dalam kotak
                    xanchor="center", yanchor="middle",
                    layer="above"
                )
            )
        except Exception as e:
            print(f"Gagal memproses {file_name}: {e}")

# Finalisasi Layout & Anotasi Tengah

In [ ]:
# 4. Pengaturan Layout
fig.update_layout(
    template="plotly_white",
    polar=dict(
        hole=0.1, # Ukuran lubang tengah
        bgcolor='white',
        radialaxis=dict(visible=False, range=[0, 220], showgrid=False),
        angularaxis=dict(
            tickvals=angles,
            ticktext=[""]*17,
            rotation=90,
            direction='clockwise',
            showgrid=False,
            showline=False
        )
    ),
    annotations=[
        dict(
            text="The length of each segments indicates<br>represents progress of<br><b>SDGs target achieved</b><br>within each goal for Jakarta In 2025",
            showarrow=False,
            x=0.5, y=0.5,
            font=dict(size=10, family="Arial")
        )
    ],
    showlegend=False,
    width=850,
    height=850,
    margin=dict(l=50, r=50, t=50, b=50)
)


# Tentukan skala HD (misalnya scale=3 atau scale=4 untuk memperbesar resolusi piksel)
fig.show(config={
    'toImageButtonOptions': {
        'format': 'png', # Format gambar
        'filename': 'sdgs_radar_chart',
        'height': 400,
        'width': 400,
        'scale': 3 # Semakin tinggi angkanya (misal 3 atau 4), semakin tajam dan HD resolusinya
    }
})